In [1]:
import rpy2.robjects.packages as rpackages
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import pandas as pd
import tqdm

import os
import sqlite3
import platform

pandas2ri.activate()

rpackages.importr('DBI')
rpackages.importr('lme4')
rpackages.importr('DT')


rpy2.robjects.packages.Package as a <module 'DT'>

In [33]:

def extract_model_summary(model_name, summary_name):

    # 1. Fixed Effects
    fixed_effects = ro.r(f'as.data.frame({summary_name}$coefficients)')
    fixed_effects_df = pandas2ri.rpy2py(fixed_effects)
    #print(fixed_effects_df)
    fixed_effects_df.columns = ["Estimate", "Std. Error", "t value", "Pr(>|t|)"]

    # # 2. Random Effects
    # random_effects = ro.r(f'as.data.frame(VarCorr({model_name}))')
    # random_effects_df = pandas2ri.rpy2py(random_effects)
    # #print("\nRandom Effects (Variance and Std. Dev by Group):\n", random_effects_df)

    # 3. Residuals
    residuals = ro.r(f'as.data.frame({summary_name}$residuals)')
    residuals_df = pandas2ri.rpy2py(residuals)
    #print("\nResiduals:\n", residuals_df)

    # 4. Model Fit Statistics
    aic = ro.r(f'AIC({model_name})')[0]
    bic = ro.r(f'BIC({model_name})')[0]
    log_likelihood = ro.r(f'logLik({model_name})')[0]
    warnings = ro.r('warnings()') 
    fit_stats_df = pd.DataFrame({
        'AIC': [aic],
        'BIC': [bic],
        'Log-Likelihood': [log_likelihood],
        'Warnings': [warnings]
    })

    # # 5. Variance-Covariance Matrix of Random Effects
    # var_cov_matrix = ro.r(f'as.data.frame({summary_name}$varcor)')
    # var_cov_matrix_df = pandas2ri.rpy2py(var_cov_matrix)
    
    return fixed_effects_df, fit_stats_df

def execute_r_lmer_model(fit_formula, data_frame_name, fit_name, fit_summary_name):
    ro.r(f'''
    {fit_name} <- lm({fit_formula}, data={data_frame_name})
    {fit_summary_name} <- summary({fit_name})
    ''')
    #print(f"Fitting model: {fit_formula}")
    
    fixed_effects, fit_stats = extract_model_summary(fit_name, fit_summary_name)

    return fixed_effects, fit_stats 

#Given a model id, load the dataset for it in r, transform the data and run the model and return the results

def fit_surprisal_model(filter_model_id):
    ro.r(f'''
           RESULTS_DB_PATH <- "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/results.db"
            results_db <- dbConnect(RSQLite::SQLite(), RESULTS_DB_PATH)

            baseline_df <- dbGetQuery(results_db,"With RESULT_TABLE AS (
                    WITH SPRT_AVG AS (
                    SELECT SPRTNaturalStories.StoryWordID, avg(SPRTNaturalStories.RT) as Avg_RT
                    FROM SPRTNaturalStories
                    GROUP BY SPRTNaturalStories.StoryWordID
                    )

                    SELECT ModelSurprisalScores.ModelID, ModelSurprisalScores.StoryWordID,
                    Story.CorpusID, Story.StoryID,
                    WordDetails.CharacterLength, WordDetails.LogFrequencies,  
                    ModelSurprisalScores.SurprisalScore,
                    SPRT_AVG.Avg_RT,

                    LAG(WordDetails.CharacterLength, 1) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev_len,
                    LAG(WordDetails.LogFrequencies, 1) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev_freq,
                    LAG(ModelSurprisalScores.SurprisalScore, 1) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev_surp,

                    LAG(WordDetails.CharacterLength, 2) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev2_len,
                    LAG(WordDetails.LogFrequencies, 2) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev2_freq,
                    LAG(ModelSurprisalScores.SurprisalScore, 2) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev2_surp,

                    LAG(WordDetails.CharacterLength, 3) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev3_len,
                    LAG(WordDetails.LogFrequencies, 3) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev3_freq,
                    LAG(ModelSurprisalScores.SurprisalScore, 3) OVER (PARTITION BY Story.StoryID ORDER BY Story.WordID) as prev3_surp

                    from ModelSurprisalScores
                    JOIN Story on Story.StoryWordID = ModelSurprisalScores.StoryWordID
                    JOIN WordDetails on WordDetails.WordUID = Story.WordUID
                    JOIN SPRT_AVG on SPRT_AVG.StoryWordID = ModelSurprisalScores.StoryWordID
                    Where ModelSurprisalScores.ModelID = {filter_model_id} 
                    ) 
                    SELECT * FROM RESULT_TABLE 
                    WHERE prev3_len IS NOT NULL
                
            ")
            '''
            )
    
    
    ro.r('''
        baseline_df$LogRT <- log(baseline_df$Avg_RT)

        baseline_df$CharacterLength_c <- scale(baseline_df$CharacterLength)
        #Should I scale Log Frequencies(?)
        baseline_df$LogFrequencies_c <- scale(baseline_df$LogFrequencies)

        baseline_df$prev_len_c <- scale(baseline_df$prev_len)
        baseline_df$prev_freq_c <- scale(baseline_df$prev_freq)
        baseline_df$prev_surp_c <- scale(baseline_df$prev_surp)

        baseline_df$prev2_len_c <- scale(baseline_df$prev2_len)
        baseline_df$prev2_freq_c <- scale(baseline_df$prev2_freq)
        baseline_df$prev2_surp_c <- scale(baseline_df$prev2_surp)
         
        baseline_df$prev3_len_c <- scale(baseline_df$prev3_len)
        baseline_df$prev3_freq_c <- scale(baseline_df$prev3_freq)
        baseline_df$prev3_surp_c <- scale(baseline_df$prev3_surp)
         

        baseline_df$SurprisalScore_c <- scale(baseline_df$SurprisalScore)
        data_size <- nrow(baseline_df)
        ''')
    
    data_frame_name = "baseline_df"

    fit_name_m = "fit_model"
    fit_summary_name_m = "fit_model_summary"

    fixed_effects_m, fit_stats_m = execute_r_lmer_model(fit_formula_m, data_frame_name, fit_name_m, fit_summary_name_m)

    return fixed_effects_m, fit_stats_m


def compile_results_as_df(model_id, fixed_effects_df, fit_stats_df, fit_stats_b):
    
    data_size = ro.r('data_size')[0]
    results_df_row = {
        "ModelID" : model_id,
        "condition_model_formula" : fit_formula_m,
        "Log-Likelihood" : fit_stats_df['Log-Likelihood'].values[0],
        "Coefficient for Surprisal Score" : fixed_effects_df.loc['SurprisalScore_c', 'Estimate'],
        "Delta Log-Likelihood" : fit_stats_df['Log-Likelihood'].values[0] - fit_stats_b['Log-Likelihood'].values[0],
        "AIC" : fit_stats_df['AIC'].values[0],
        "BIC" : fit_stats_df['BIC'].values[0],
        "Data Size" : data_size,
        "baseline_model_formula" : fit_formula_b,
        'Fixed Effects': fixed_effects_df.to_html(),
    }
        
    #results_df = pd.DataFrame(results_df_row, index=[0])

    return results_df_row



def append_or_overwrite_csv(file_path, new_row):
    # Read the existing CSV file
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        # If the file does not exist, create a new DataFrame
        df = pd.DataFrame(columns=new_row.keys())
    
    # Check if the row already exists
    # This assumes that the DataFrame has a unique identifier column called 'id'
    if 'ModelID' in new_row:  # Ensure that there's a unique identifier
        existing_row_index = df[df['ModelID'] == new_row['ModelID']].index
        
        if not existing_row_index.empty:
            # If the row exists, update it
            df.loc[existing_row_index[0]] = new_row  # Update the first matching row
        else:
            # If the row does not exist, append it
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        print("New row must contain a unique identifier under the key 'ModelID'.")

    # Save the updated DataFrame back to CSV
    df.to_csv(file_path, index=False)



In [34]:

if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'#Given a set of details, return a dataframe compiled with the details

RESULTS_ROOT = os.path.join(ROOT, "results")
SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)

#MODEL_LIST_QUERY = "SELECT ModelID from ModelSurprisalScores"

MODEL_LIST_QUERY = '''
SELECT DISTINCT ModelSurprisalScores.ModelID, Model.OutputFolderName, Model.BatchSize, Model.Dataset, Model.Seed, Model.MaskType FROM ModelSurprisalScores
JOIN Model on Model.ModelID = ModelSurprisalScores.ModelID
WHERE Model.NumLayers = 6 
AND ((Model.EchoicMemory=10 AND Model.MaskType="exponential_new" AND Model.MaskDecayRate=2) OR (Model.MaskType="Non" AND Model.CurriculumLearning=False)) 
AND Model.Dataset in ("babylm_full_bpe_8k", "babylm_full_bpe_100M_8k")  
AND Model.ModelID not in (5496427, 8456913)
ORDER BY Seed, BatchSize, Dataset, MaskType
'''


model_id_list = pd.read_sql_query(MODEL_LIST_QUERY, conn)['ModelID'].unique().tolist()
print(len(model_id_list), model_id_list)



62 [8465733, 8465085, 6892214, 6839425, 8117279, 8117319, 8465604, 8465082, 6892212, 6839403, 8111923, 8111939, 8465611, 8465091, 6892220, 6839430, 8111928, 8111942, 8465607, 8465086, 6892216, 6839426, 8111925, 8111941, 8465610, 8465090, 6892219, 6839429, 8117282, 8117325, 8096895, 8465077, 6892222, 6681944, 8098216, 8111938, 8464992, 8456915, 8465605, 8465084, 6892213, 6839424, 8111924, 8111940, 8465609, 8465089, 6892218, 6839428, 8117281, 8117322, 8465612, 8465093, 6892221, 6839431, 8117283, 8117326, 8465608, 8465087, 6892217, 6839427, 8117280, 8117321]


In [36]:

data_frame_name = "baseline_df"
fit_name_b = "fit_baseline"
fit_summary_name_b = "fit_summary_baseline"
fit_stats_b = None

# fixed_effects_b, random_effects_b, fit_stats_b, var_cov_matrix_b = execute_r_lmer_model(fit_formula_b, data_frame_name, fit_name_b, fit_summary_name_b)

#fit_formula_b0 = "LogRT ~ CharacterLength_c + (1 | WorkerID) + (1 | POSTag)"
# fit_formula_b = "LogRT ~ CharacterLength_c + LogFrequencies_c + (1 | WorkerID) + (1 | POSTag)"
# fit_formula_m = "LogRT ~ CharacterLength_c + LogFrequencies_c + SurprisalScore_c + (1 | WorkerID) + (1 | POSTag)"

fit_formula_b = "LogRT ~ CharacterLength_c + prev_len_c + prev2_len_c + prev3_len_c + LogFrequencies_c + prev_freq_c + prev2_freq_c + prev3_freq_c"
fit_formula_m = "LogRT ~ CharacterLength_c + prev_len_c + prev2_len_c + prev3_len_c + LogFrequencies_c + prev_freq_c + prev2_freq_c + prev3_freq_c + SurprisalScore_c + prev_surp_c + prev2_surp_c + prev3_surp_c"

fit_formula_b = "LogRT ~ CharacterLength_c*LogFrequencies_c + prev_len_c*prev_freq_c + prev2_len_c*prev2_freq_c + prev3_len_c*prev3_freq_c"
fit_formula_m = "LogRT ~ SurprisalScore_c + prev_surp_c + prev2_surp_c + prev3_surp_c + CharacterLength_c*LogFrequencies_c + prev_len_c*prev_freq_c + prev2_len_c*prev2_freq_c + prev3_len_c*prev3_freq_c"

write_path = "surprisal_analysis_results_v4.csv"

#results_df = pd.DataFrame()

def verify_model_already_processed(model_id, write_path):
    try:
        df = pd.read_csv(write_path)
        return df['ModelID'].tolist()
    except FileNotFoundError:
        return []

for model_id in tqdm.tqdm(model_id_list):
    if model_id in verify_model_already_processed(model_id, write_path):
        continue
    try:    
        fixed_effects_m, fit_stats_m = fit_surprisal_model(model_id)
        if fit_stats_b is None:
            print(f"Baseline model not fit yet. Fitting baseline model for model id {model_id}")
            fixed_effects_b, fit_stats_b = execute_r_lmer_model(fit_formula_b, data_frame_name, fit_name_b, fit_summary_name_b)

        append_row = compile_results_as_df(model_id, fixed_effects_m, fit_stats_m, fit_stats_b)
        #print(append_row['ModelID'], append_row['Delta Log-Likelihood'])
        append_or_overwrite_csv(write_path, append_row)
    except Exception as e:
        print(f"Error for model id {model_id}: {e}")
        continue
        



  0%|          | 0/62 [00:00<?, ?it/s]/tmp/ipykernel_2324820/3220730036.py:168: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
  2%|▏         | 1/62 [00:00<00:18,  3.26it/s]

Baseline model not fit yet. Fitting baseline model for model id 8465733


100%|██████████| 62/62 [00:15<00:00,  3.88it/s]


In [18]:

fit_name_b0 = "fit_baseline"
fit_summary_name_b0 = "fit_summary_baseline"
fit_formula_b0 = "LogRT ~ CharacterLength_c + (1 | WorkerID) + (1 | POSTag)"
fixed_effects_b0, random_effects_b0, fit_stats_b0, var_cov_matrix_b0 = execute_r_lmer_model(fit_formula_b0, data_frame_name, fit_name_b0, fit_summary_name_b0)

fit_formula_b = "LogRT ~ CharacterLength_c + LogFrequencies_c + (1 | WorkerID) + (1 | POSTag)"
fixed_effects_b, random_effects_b, fit_stats_b, var_cov_matrix_b = execute_r_lmer_model(fit_formula_b, data_frame_name, fit_name_b, fit_summary_name_b)


In [22]:
print("B0 Model", fit_stats_b0["Log-Likelihood"], "\n B Model", fit_stats_b["Log-Likelihood"], "\n Delta", fit_stats_b["Log-Likelihood"] - fit_stats_b0["Log-Likelihood"])



B0 Model 0   -172359.238659
Name: Log-Likelihood, dtype: float64 
 B Model 0   -171372.22167
Name: Log-Likelihood, dtype: float64 
 Delta 0    987.016988
Name: Log-Likelihood, dtype: float64


In [30]:
print("B0 Model", fit_stats_b0["AIC"], "\n B Model", fit_stats_b["AIC"], "\n Delta", fit_stats_b["AIC"] - fit_stats_b0["AIC"])
print("\n \n")

print("B0 Model", fit_stats_b0["BIC"], "\n B Model", fit_stats_b["BIC"], "\n Delta", fit_stats_b["BIC"] - fit_stats_b0["BIC"])

B0 Model 0    344728.477317
Name: AIC, dtype: float64 
 B Model 0    342756.44334
Name: AIC, dtype: float64 
 Delta 0   -1972.033977
Name: AIC, dtype: float64

 

B0 Model 0    344786.735017
Name: BIC, dtype: float64 
 B Model 0    342826.35258
Name: BIC, dtype: float64 
 Delta 0   -1960.382437
Name: BIC, dtype: float64


In [23]:
fixed_effects_b0

,Estimate,Std. Error,t value
(Intercept),5.737890,0.025029,229.249971
CharacterLength_c,0.026544,0.000395,67.155594


In [24]:
fixed_effects_b

,Estimate,Std. Error,t value
(Intercept),5.738187,0.023070,248.726947
CharacterLength_c,0.012079,0.000512,23.614224
LogFrequencies_c,-0.027003,0.000607,-44.465857
